In [1]:
!pip install psycopg2-binary
!pip install -qU sentence-transformers

In [2]:
#загрузка файла all_filtered_chunks.json в БД

import json
import psycopg2
from psycopg2.extras import execute_batch

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 5433,
    "database": "ragdatabase",
    "user": "raguser",
    "password": "ragpassword"
}

DOCUMENT_NAMES = {
    0: "01_Elicont_100___1_____04",
    1: "02_Elicont_100_1__2____09_07",
    2: "03_Elicont_100_2__3_____",
    3: "04_Elicont_200___1_____22",
    4: "05_Elicont_200_1__2____14_10",
    5: "06_Elicont_200_2__3_____",
    6: "07_",
    7: "08_",
    8: "09_",
    9: "10_",
    10: "11_",
    11: "12_",
    12: "13_",
    13: "14_",
    14: "15_",
    15: "16_",
    16: "17_",
    17: "18_"
}

JSON_PATH = "all_filtered_chunks.json"


def main():
    # 1. Загружаем JSON
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        all_chunks = json.load(f)

    # 2. Проверка структуры
    if not isinstance(all_chunks, list):
        raise ValueError("Ожидался список списков в all_filtered_chunks.json")

    # 3. Готовим записи для вставки
    rows_to_insert = []

    for doc_idx, doc_chunks in enumerate(all_chunks):
        if not isinstance(doc_chunks, list):
            raise ValueError(f"Элемент с индексом {doc_idx} не является списком чанков")

        source_doc = DOCUMENT_NAMES.get(doc_idx)
        if source_doc is None:
            raise ValueError(f"Для индекса {doc_idx} нет имени документа в DOCUMENT_NAMES")

        for chunk in doc_chunks:
            if not isinstance(chunk, str):
                continue

            chunk = chunk.strip()
            if not chunk:
                continue

            rows_to_insert.append((source_doc, chunk))

    print(f"Подготовлено {len(rows_to_insert)} чанков для вставки")

    # 4. Подключаемся к БД
    conn = psycopg2.connect(**DB_CONFIG)

    try:
        with conn:
            with conn.cursor() as cur:
                # 5. Создаём расширение и таблицу
                cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

                cur.execute("""
                    CREATE TABLE IF NOT EXISTS chunks (
                        id SERIAL PRIMARY KEY,
                        source_doc TEXT NOT NULL,
                        chunk TEXT NOT NULL,
                        vector VECTOR(768)
                    );
                """)

                # Если хочешь перед новой загрузкой очищать таблицу, раскомментируй:
                cur.execute("TRUNCATE TABLE chunks RESTART IDENTITY;")

                # 6. Вставляем данные
                execute_batch(
                    cur,
                    """
                    INSERT INTO chunks (source_doc, chunk)
                    VALUES (%s, %s)
                    """,
                    rows_to_insert,
                    page_size=500
                )

        print("Данные успешно занесены в БД")

    finally:
        conn.close()


if __name__ == "__main__":
    main()

Подготовлено 1284 чанков для вставки
Данные успешно занесены в БД


In [3]:
import zipfile
from pathlib import Path
from sentence_transformers import SentenceTransformer

# загрузка ретривера
zip_path = Path("e5_custom (2).zip")
extract_to = Path("e5_custom")

extract_to.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

print("Готово:", extract_to.resolve())

model = SentenceTransformer("e5_custom/kaggle/working/e5_custom")

Готово: C:\Users\4789201\tulahack\tulahack2026\backend\e5_custom


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
# заполнение колонки vector в БД только для записей из 01 pdf

import psycopg2

def to_pgvector(vec):
    return "[" + ",".join(str(float(x)) for x in vec) + "]"

TARGET_DOC = "01_Elicont_100___1_____04"

conn = psycopg2.connect(
    host="127.0.0.1",
    port=5433,
    database="ragdatabase",
    user="raguser",
    password="ragpassword"
)

cur = conn.cursor()

cur.execute("""
    SELECT id, source_doc, chunk
    FROM chunks
    WHERE vector IS NULL
      AND source_doc = %s
""", (TARGET_DOC,))

dataset = cur.fetchall()
print(f"Найдено {len(dataset)} чанков для {TARGET_DOC}")

ids = []
texts = []

for chunk_id, source_doc, chunk_text in dataset:
    ids.append(chunk_id)
    texts.append(f"passage: {source_doc}. {chunk_text}")

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

updates = [
    (to_pgvector(emb), chunk_id)
    for chunk_id, emb in zip(ids, embeddings)
]

cur.executemany("""
    UPDATE chunks
    SET vector = %s::vector
    WHERE id = %s
""", updates)

conn.commit()
# cur.close()
# conn.close()

print(f"Эмбеддинги для {TARGET_DOC} записаны")

Найдено 79 чанков для 01_Elicont_100___1_____04


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Эмбеддинги для 01_Elicont_100___1_____04 записаны


In [6]:
cur = conn.cursor()

cur.execute("""
    SELECT id, source_doc, chunk, vector
    FROM chunks
    LIMIT 5
""")

rows = cur.fetchall()

for row in rows:
    print(row)

(2, '01_Elicont_100___1_____04', "2 Проектная компоновка Контроллера « El-100»\n\nПроектная компоновка позволяет создавать Контроллеры «El -100 » со следующей аппаратной структурой:\n\n-модуль  процессора  (далее -Процессор ) ,  имеющий  подсистему  управления  вводом -выводом  информации,  подсистему  выполнения  загруженной  технологической  программы  и сетевую  подсистему  для  информационной  связи  с  другими  контроллерами  и  со  средствами системы представления информации и архивирования в ПТК;\n\n-набор многоканальных устройств связи с объектом управления (модулей УСО), обеспечивающих обмен информацией процессора с объектом управления по физическим линиям. Набор модулей УСО определяется проектным путём. С контроллерной сетью Процессора модули УСО связываются при помощи модулей интерфейсной связи;\n\n-кроссовые средства в виде полевых адаптеров для подключения кабелей связи от объекта управления к модулям УСО;\n\n- -модульные элементы системы электропитания Контроллера «El -10